In [22]:
# pip install --upgrade langgraph
# pip install --upgrade langchain

In [24]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer


class State(TypedDict):
    topic: str
    joke: str


def generate_joke(state: State):
    writer = get_stream_writer()
    writer({"status": "thinking of a joke..."})
    return {"joke": f"Why did the {state['topic']} go to school? To get a sundae education!"}

graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_edge(START, "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)



In [26]:
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["updates", "custom"],
    version="v2",
):
    print("\n\n CHUNK :", chunk)
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node {node_name} updated: {state}")
    elif chunk["type"] == "custom":
        print(f"Status: {chunk['data']['status']}")



 CHUNK : {'type': 'custom', 'ns': (), 'data': {'status': 'thinking of a joke...'}}
Status: thinking of a joke...


 CHUNK : {'type': 'updates', 'ns': (), 'data': {'generate_joke': {'joke': 'Why did the ice cream go to school? To get a sundae education!'}}}
Node generate_joke updated: {'joke': 'Why did the ice cream go to school? To get a sundae education!'}


In [28]:
inputs = {"topic": "ice cream"}

for chunk in graph.stream(inputs, stream_mode="updates", version="v2"):
    print(chunk)
    print("-----------------")
    print(chunk["type"])  # "updates"
    print(chunk["ns"])    # ()
    print(chunk["data"])  # {"node_name": {"key": "value"}}

{'type': 'updates', 'ns': (), 'data': {'generate_joke': {'joke': 'Why did the ice cream go to school? To get a sundae education!'}}}
-----------------
updates
()
{'generate_joke': {'joke': 'Why did the ice cream go to school? To get a sundae education!'}}


In [29]:
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)  # {"node_name": {"key": "value"}}

{'generate_joke': {'joke': 'Why did the ice cream go to school? To get a sundae education!'}}


In [34]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer


class State(TypedDict):
    topic: str
    joke: str


def generate_joke(state: State):
    writer = get_stream_writer()
    writer({"progress": "40", "status": "thinking of a joke..."})
    writer({"progress": "90", "status": "almost done..."})
    return {"joke": f"Why did the {state['topic']} go to school? To get a sundae education!"}

graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_edge(START, "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)


In [35]:
for part in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["values", "updates", "messages", "custom"],
    version="v2",
):
    if part["type"] == "values":
        # ValuesStreamPart — full state snapshot after each step
        print(f"State: topic={part['data']['topic']}")
    elif part["type"] == "updates":
        # UpdatesStreamPart — only the changed keys from each node
        for node_name, state in part["data"].items():
            print(f"Node `{node_name}` updated: {state}")
    elif part["type"] == "messages":
        # MessagesStreamPart — (message_chunk, metadata) from LLM calls
        msg, metadata = part["data"]
        print(msg.content, end="", flush=True)
    elif part["type"] == "custom":
        # CustomStreamPart — arbitrary data from get_stream_writer()
        print(f"Progress: {part['data']['progress']}%")
        print(f"Status: {part['data']['status']}")

State: topic=ice cream
Progress: 40%
Status: thinking of a joke...
Progress: 90%
Status: almost done...
Node `generate_joke` updated: {'joke': 'Why did the ice cream go to school? To get a sundae education!'}
State: topic=ice cream
